# Apuntes: Entrenamiento de un perceptrón simple en TensorFlow 2.x con MNIST
### 1️⃣ Carga y preparación de datos

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0

x_test = x_test.astype("float32") / 255.0

y_train = tf.keras.utils.to_categorical(y_train, 10)

y_test = tf.keras.utils.to_categorical(y_test, 10)



#### Explicación:

tf.keras.datasets.mnist.load_data() carga las imágenes y etiquetas separadas en train y test.

Las imágenes se normalizan de 0–255 a 0–1 (astype("float32") / 255.0) para que la red aprenda más rápido.

Las etiquetas se convierten en one-hot encoding (to_categorical) para que la clase correcta sea representada con un 1 en la posición correspondiente y 0 en el resto.

### 2️⃣ Aplanar imágenes para perceptrón
x_train = x_train.reshape(-1, 28*28)

x_test = x_test.reshape(-1, 28*28)


#### Explicación:

Cada imagen de 28×28 se aplana a un vector de 784 elementos.

-1 mantiene todas las imágenes (batch) automáticamente.

Resultado: x_train.shape = (60000, 784).

### 3️⃣ Inicialización de pesos y sesgo
n_inputs = 28*28

n_outputs = 10

W = tf.Variable(tf.random.normal([n_inputs, n_outputs], stddev=0.1))

b = tf.Variable(tf.zeros([n_outputs]))


#### Explicación:

W → matriz de pesos (784, 10) conectando cada píxel con cada clase.

b → vector de sesgos (10,) que ajusta la activación de cada clase.

Inicializamos W con valores pequeños aleatorios y b en cero.

Analogía:

W indica la importancia de cada píxel para cada dígito.

b permite que la neurona se active incluso si los píxeles son 0.

### 4️⃣ Definir optimizador y función de pérdida
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)
loss_fn = tf.keras.losses.CategoricalCrossentropy()


#### Explicación:

Optimizador (SGD): actualiza los pesos y sesgos en la dirección que minimiza la pérdida.

Función de pérdida (categorical crossentropy): mide qué tan lejos están las predicciones de las etiquetas reales, considerando también la confianza de la predicción.

Ejemplo: predecir correctamente con 90% → pérdida baja.

Predecir correctamente con 60% → pérdida más alta.

### 5️⃣ Bucle de entrenamiento
epochs = 5

batch_size = 128

num_batches = x_train.shape[0] // batch_size


#### Explicación:

epochs → número de pasadas completas sobre todo el dataset (5).

batch_size → tamaño de cada mini-batch (128).

num_batches → número de batches por epoch (~468).

### 6️⃣ Mezclar datos al inicio de cada epoch
for epoch in range(epochs):

    idx = np.random.permutation(x_train.shape[0])

    x_train_shuffled = x_train[idx]

    y_train_shuffled = y_train[idx]


#### Explicación:

np.random.permutation genera un orden aleatorio de índices para barajar las imágenes y etiquetas.

Esto evita que la red siempre vea las imágenes en el mismo orden.

x_train_shuffled y y_train_shuffled contienen las mismas imágenes y etiquetas pero reordenadas.

### 7️⃣ Inicializar acumulador de pérdida
epoch_loss = 0


Se resetea la pérdida acumulada del epoch para luego calcular la pérdida promedio.

### 8️⃣ Procesar cada mini-batch
for i in range(num_batches):

    x_batch = x_train_shuffled[i*batch_size:(i+1)*batch_size]

    y_batch = y_train_shuffled[i*batch_size:(i+1)*batch_size]


#### Explicación:

Se extraen lotes de tamaño batch_size para procesar en el forward pass y actualizar los pesos.

Cada imagen sigue emparejada con su etiqueta real.

### 9️⃣ Forward pass y cálculo de pérdida con GradientTape
with tf.GradientTape() as tape:

    logits = tf.matmul(x_batch, W) + b

    y_pred = tf.nn.softmax(logits)
    
    loss = loss_fn(y_batch, y_pred)


#### Explicación línea por línea:

logits = tf.matmul(x_batch, W) + b

Multiplica las imágenes por los pesos y suma el sesgo → predicciones sin normalizar (logits).

y_pred = tf.nn.softmax(logits)

Convierte logits en probabilidades por clase.

La etiqueta predicha es la que tiene la probabilidad más alta.

loss = loss_fn(y_batch, y_pred)

Calcula la pérdida de todo el batch.

Mide cuán lejos están las predicciones de las etiquetas reales, incluyendo la confianza.

### 10️⃣ Backpropagation: cálculo de gradientes
gradients = tape.gradient(loss, [W, b])


### Explicación:

Calcula los gradientes de la pérdida respecto a W y b.

grad_W → (784, 10), indica cuánto ajustar cada peso.

grad_b → (10,), indica cuánto ajustar cada sesgo.

Los gradientes consideran todas las clases y todas las imágenes del batch, no solo la clase ganadora.

### 11️⃣ Actualización de pesos y sesgos
optimizer.apply_gradients(zip(gradients, [W, b]))


Explicación:

zip(gradients, [W, b]) empareja cada gradiente con su variable.

apply_gradients actualiza W y b según los gradientes y el learning rate:

W = W - \text{learning_rate} \cdot grad_W b = b - \text{learning_rate} \cdot grad_b
### 12️⃣ Acumular pérdida del epoch
epoch_loss += loss.numpy()


Convierte el tensor loss a número real y lo suma al acumulador epoch_loss.

Permite calcular la pérdida promedio al final del epoch.

### 13️⃣ Mostrar progreso
print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/num_batches:.4f}")


Muestra el número del epoch y la pérdida promedio del epoch.

epoch+1 → hace que el conteo comience en 1.

epoch_loss/num_batches → pérdida promedio de todos los mini-batches.

### 14️⃣ Evaluación del modelo
logits_test = tf.matmul(x_test, W) + b
y_pred_test = tf.nn.softmax(logits_test)
accuracy = np.mean(np.argmax(y_pred_test.numpy(), axis=1) == np.argmax(y_test, axis=1))
print(f"Test accuracy: {accuracy:.4f}")


#### Explicación:

Multiplicamos las imágenes de test por los pesos entrenados y sumamos los sesgos.

Aplicamos softmax para obtener probabilidades de cada clase.

argmax obtiene la clase con mayor probabilidad.

Comparamos con las etiquetas reales y tomamos la media → accuracy.

True → 1, False → 0

Resultado: porcentaje de imágenes correctamente clasificadas.

#### 🔑 Resumen conceptual final

Las imágenes se aplanan y normalizan.

Los pesos y sesgos se inicializan aleatoriamente y se van ajustando batch por batch mediante gradient descent.

La pérdida mide qué tan lejos están las predicciones de la realidad, considerando también la confianza.

Los gradientes calculan cómo ajustar cada peso y sesgo para mejorar la predicción.

Al final de los epochs, los pesos y sesgos finales permiten evaluar la red con datos de test y obtener la accuracy.